# 3. Backtest and evaluation

The whole method re-run through history, month by month, and scored against the
rule that was written down before it ran. Stage gates four and five.

Run `forecast backtest` and `forecast evaluate` before this notebook.

In [ ]:
import pandas as pd

from economic_regime_forecasting import pipeline_gates
from economic_regime_forecasting.configuration.registry import load_registries
from economic_regime_forecasting.configuration.run_settings import (
    ARTIFACTS,
    DEFAULT_RUN_SETTINGS,
)
from economic_regime_forecasting.data.cache import ArtifactStore
from economic_regime_forecasting.evaluation.calibration import assess_calibration
from economic_regime_forecasting.evaluation.verdict import load_decision_rule
from economic_regime_forecasting.reporting import figures

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

settings = DEFAULT_RUN_SETTINGS
registry, indicators = load_registries()
artifacts = ArtifactStore(settings.cache.models)

results = artifacts.read_table(ARTIFACTS.backtest_results)
metrics = artifacts.read_table(ARTIFACTS.evaluation_metrics)
verdicts = artifacts.read_table(ARTIFACTS.verdicts)

print(
    f"{len(results):,} forecasts from {results['forecast_date'].min():%Y-%m} "
    f"to {results['forecast_date'].max():%Y-%m}, "
    f"{int(results['realised_outcome'].notna().sum()):,} of them resolved"
)
# Which run is this? Since ADR 0008 the pipeline default is the honest
# configuration, and the headline tables in docs/RESULTS.md are the shipped
# one, so a rendered notebook must say which it is showing.
print(
    "configuration_hash on these artifacts: "
    f"{sorted(set(results['configuration_hash']))}; "
    f"these settings are {settings.configuration_hash()}"
)

## The rule the results are being judged against

Read from the committed pre-registration rather than restated here, so that
loosening a threshold after seeing results would show up as a change to a file
under version control.

In [ ]:
rule = load_decision_rule()
pd.DataFrame(
    [{"gate": name, "requirement": body["requirement"]} for name, body in rule["gates"].items()]
)

## Gate 4

In [ ]:
print(pipeline_gates.gate_four_backtest(results).describe())

## Every metric, per indicator and horizon

`brier_skill_score` is the headline. Positive means the model beat an expanding
climatology; zero means it matched it; negative means the base rate would have
been better. `mean_distance_to_stationary` is how much regime information was
left at that horizon.

In [ ]:
metrics.round(4)

## Skill by indicator, one chart per horizon

The pattern to look for is skill concentrated at the short horizon and decaying,
which is what regime persistence predicts.

In [ ]:
for horizon in settings.forecast_horizons_in_months:
    display(figures.plot_skill_by_indicator(metrics, horizon))

## Calibration

A forecast of seventy percent should be right about seventy percent of the time.
Discrimination and calibration are different virtues, and under Brier scoring a
badly calibrated model with good discrimination loses to a well calibrated one
with none.

In [ ]:
reliability_tables = {}
calibration_errors = {}
for horizon in settings.forecast_horizons_in_months:
    scored = results[(results["horizon_months"] == horizon) & results["realised_outcome"].notna()]
    report = assess_calibration(
        scored["predicted_probability"].to_numpy(),
        scored["realised_outcome"].to_numpy(),
    )
    reliability_tables[horizon] = report.table()
    calibration_errors[horizon] = report.expected_calibration_error
    print(f"{horizon // 12} year: {report.describe()}")

figures.plot_reliability(reliability_tables, calibration_errors)

## The verdict per horizon

Each horizon ships the model only if all five gates hold. A horizon that fails any
gate ships the climatological base rate and names the gate that failed. The
interval is a moving-block bootstrap with blocks as long as the horizon, so
overlapping monthly forecasts are not counted as independent evidence.

In [ ]:
verdicts.round(4)

In [ ]:
figures.plot_skill_by_horizon(verdicts)

## Gate 5

In [ ]:
print(
    pipeline_gates.gate_five_evaluation(verdicts, settings.forecast_horizons_in_months).describe()
)

## Every gate, in one table

Written by `forecast check-gates`. This is the whole pipeline's verdict on itself.

In [ ]:
artifacts.read_table(ARTIFACTS.gate_reports)